# Test: an arbitrary vertical section with `xroms.isoslice`

`xroms.isoslice` interpolates a variable to the places where an `iso_array` equals given values along one grid axis. Its documented uses are constant depth, constant longitude or latitude, and isosurfaces of another field. This notebook tests whether it can also give a section along a line between two arbitrary lon/lat points, by choosing an `iso_array` that is zero along that line: the signed cross-track distance. The result is compared with `xroms.interpll`, which takes the points directly (see `eccofs_avg_best_nj_transect_xroms.ipynb`).

Setup is the same as in that notebook: open the whole-column best-estimate store (`eccofs-avg-best-nosub`, the better chunking for sections: see the README, "Chunking"), `xroms.subset` a box, build the xroms dataset.

In [ ]:
# %pip install -q "xroms==0.6.2" "xgcm==0.8.1" xesmf hvplot datashader icechunk zarr

In [ ]:
import warnings

import hvplot.xarray  # noqa: F401
import icechunk
import numpy as np
import xarray as xr
import xroms
import zarr

warnings.filterwarnings("ignore", module="xgcm")
zarr.config.set({"async.concurrency": 32})

# whole-column chunks (all 50 levels of a tile): the better layout for sections. For maps and time series use eccofs-avg-best
URL = "https://data.source.coop/ocean-icechunks/noaa-eccofs/eccofs-avg-best-nosub"
SOURCE = "s3://noaa-nos-eccofs-pds/"   # public bucket holding the NetCDF bytes

START = (-74.30, 39.30)   # lon, lat: off Atlantic City, NJ
END = (-72.00, 37.20)
WHEN = "2026-11-15 12:00"

In [ ]:
repo = icechunk.Repository.open(
    icechunk.http_storage(URL),
    authorize_virtual_chunk_access=icechunk.containers_credentials({SOURCE: icechunk.s3_anonymous_credentials()}),
)
ds = xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks={})

box = ((ds.lon_rho > -76) & (ds.lon_rho < -70) & (ds.lat_rho > 36) & (ds.lat_rho < 41)).load()
rows, cols = np.flatnonzero(box.any("xi_rho")), np.flatnonzero(box.any("eta_rho"))
hdims = [d for d in ds.dims if d.startswith(("xi_", "eta_"))]
region = xroms.subset(
    ds.assign_coords({d: np.arange(ds.sizes[d]) for d in hdims}),
    X=slice(cols[0], cols[-1] + 1), Y=slice(rows[0], rows[-1] + 1),
)
fields = ["temp", "u", "v", "zeta", "h", "Cs_r", "Cs_w", "pm", "pn", "angle", "f", "mask_rho", "mask_u", "mask_v", "mask_psi"]
scalars = ["Vtransform", "Vstretching", "theta_s", "theta_b", "Tcline", "hc", "rho0", "spherical"]
sub = region[fields + scalars].sel(time=WHEN, method="nearest")
sub.attrs["Conventions"] = "CF-1.4"
sub = sub.chunk({d: -1 for d in sub.dims})
rds, xgrid = xroms.roms_dataset(sub)

mask = rds.mask_rho == 1
temp = rds.temp.where(mask).load()
z_rho = rds.z_rho.load()
h = rds.h.where(mask).load()
print(dict(temp.sizes))

## Cross-track and along-track distance

Both are in km, from a local equirectangular projection about `START`. `cross` is zero on the line and changes sign across it; `along` is the distance along the line from `START`. Where the line runs through the grid, `cross` must be monotonic along the axis `isoslice` searches (here `xi`, so each grid row crosses the line once).

In [ ]:
lon0, lat0 = START
kx, ky = 111.32 * np.cos(np.deg2rad(lat0)), 110.57   # km per degree of lon, lat

def to_km(lon, lat):
    return (lon - lon0) * kx, (lat - lat0) * ky

ex, ey = to_km(*END)
L = np.hypot(ex, ey)
ux, uy = ex / L, ey / L                               # unit vector along the line

x, y = to_km(rds.lon_rho, rds.lat_rho)
cross = (-uy * x + ux * y).rename("cross").load()     # signed distance from the line
along = (ux * x + uy * y).rename("along").load()
d = cross.diff("xi_rho")
assert bool((d > 0).all() or (d < 0).all()), "cross-track distance must be monotonic along xi"
print(f"line length {L:.0f} km")

## `isoslice` at `cross = 0`

One value per grid row, so the spacing along the line is about the grid spacing divided by the cosine of the line's angle to the rows. A line running nearly parallel to the rows would cross each row far apart (or not at all); use `axis="Y"` and `cross` monotonic along `eta` for that case.

Two workarounds were needed: `iso_values` must be an array, not a list (the docstring examples pass a bare number), and fields without a vertical dimension (`h`, `along`) fail inside `isoslice`. The latter is a cf-xarray bug that `xroms.order` runs into: `da.cf.transpose("Y")` with a single name unpacks the name into characters ([cf-xarray#696](https://github.com/xarray-contrib/cf-xarray/issues/696), fix in [#697](https://github.com/xarray-contrib/cf-xarray/pull/697)). Until that is released, the 2-D fields are broadcast onto `s_rho` and the extra dimension is dropped again.

In [ ]:
def across(v):
    two_d = "s_rho" not in v.dims
    if two_d:
        v = v.reset_coords(drop=True).expand_dims(s_rho=rds.s_rho.values).assign_coords(s_rho=rds.s_rho)
    r = xroms.isoslice(v, np.array([0.0]), xgrid, iso_array=cross, axis="X")
    return r.isel(s_rho=0, drop=True) if two_d else r

sec = xr.Dataset({"temp": across(temp), "z_rho": across(z_rho), "h": across(h), "along": across(along)}).squeeze()
sec = sec.where((sec.along >= 0) & (sec.along <= L), drop=True).sortby("along")
sec = sec.set_coords(["along", "z_rho"])
print(f"{sec.sizes['eta_rho']} points, spacing {float(sec.along.diff('eta_rho').mean()):.2f} km")

In [ ]:
sec = sec.assign_coords(km=(("s_rho", "eta_rho"), np.broadcast_to(sec.along.values, sec.z_rho.shape).copy()))
sec.temp.hvplot.quadmesh(
    x="km", y="z_rho", rasterize=True, cmap="turbo", colorbar=True,
    xlabel="distance from the coast (km)", ylabel="depth (m)", title="temperature from isoslice(cross = 0)",
    frame_width=700, frame_height=350, ylim=(-1000, 5),
)

## Compare with `xroms.interpll`

`interpll` takes the line as lon/lat points and interpolates bilinearly on the sphere. Its points are placed along the same line; each is given its along-track distance with the same projection, and the `isoslice` section is interpolated to those positions level by level.

In [ ]:
lons, lats = np.linspace(START[0], END[0], 103), np.linspace(START[1], END[1], 103)
ref = xroms.interpll(temp, lons, lats, which="pairs")
px, py = to_km(lons, lats)
ref_along = ux * px + uy * py

# isoslice section interpolated to the interpll positions (restrict to where both exist)
a = sec.along.values
diff = np.stack([
    np.interp(ref_along, a, sec.temp.isel(s_rho=k).values, left=np.nan, right=np.nan) - ref.isel(s_rho=k).values
    for k in range(ref.sizes["s_rho"])
])
print(f"isoslice - interpll: mean |diff| {np.nanmean(np.abs(diff)):.3f} degC, 99th percentile {np.nanpercentile(np.abs(diff), 99):.3f}, max {np.nanmax(np.abs(diff)):.3f}")

In [ ]:
dd = xr.DataArray(diff, dims=("s_rho", "pt"), coords={"z_rho": (("s_rho", "pt"), ref.z_rho.values), "km": (("s_rho", "pt"), np.broadcast_to(ref_along, diff.shape).copy())}, name="diff")
dd.hvplot.quadmesh(
    x="km", y="z_rho", rasterize=True, cmap="RdBu_r", symmetric=True, colorbar=True,
    xlabel="distance from the coast (km)", ylabel="depth (m)", title="isoslice minus interpll (degC)",
    frame_width=700, frame_height=350, ylim=(-1000, 5),
)

## Result

`isoslice` can produce a section along an arbitrary line, with the cross-track-distance trick, and it agrees with `interpll` to within the differences printed above. It is more work, though: the cross/along fields, a monotonicity condition on the chosen axis, one point per grid row, the trimming and sorting, and two workarounds for xroms 0.6.2. `xroms.interpll` does it in one call from the endpoints, so it is the better tool for sections between two points. `isoslice` remains the right one for constant-depth, constant-latitude or constant-longitude slices and for isosurfaces.